In [10]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import os
import logging



sns.set(style="whitegrid")

In [11]:
LOG_DIR = "Data/logs"
os.makedirs(LOG_DIR, exist_ok=True)

log_file = f"{LOG_DIR}/eda_pipeline_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

logging.basicConfig(
    filename=log_file,
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logging.info("EDA Pipeline Started")

In [12]:
METADATA_PATH = r"E:\Automated-E-Commerce-Funnel-Analysis-Business-Intelligence-Pipeline\\Data\\metadata\\latest_metadata.json"

with open(METADATA_PATH, "r") as f:
    metadata = json.load(f)

logging.info(f"Loaded batch_id: {metadata['batch_id']}")

In [13]:
logging.info("Loading datasets from metadata")

orders = pd.read_csv(metadata["raw_data_paths"]["orders"])
users = pd.read_csv(metadata["raw_data_paths"]["users"])
events = pd.read_csv(metadata["raw_data_paths"]["events"])
sessions = pd.read_csv(metadata["raw_data_paths"]["sessions"])
products = pd.read_csv(metadata["raw_data_paths"]["products"])

logging.info("Datasets loaded successfully")

FileNotFoundError: [Errno 2] No such file or directory: 'Data/raw/orders/orders_2026_W21.csv'

In [ ]:
def data_quality_report(df, name):

    logging.info(f"Running data quality check for {name}")

    print("\n========================")
    print(f"{name} REPORT")
    print("========================")

    print("Shape:", df.shape)
    print("Missing Values:\n", df.isnull().sum())
    print("Duplicates:", df.duplicated().sum())
    print("Dtypes:\n", df.dtypes)

    logging.info(
        f"{name} - Shape: {df.shape}, Missing: {df.isnull().sum().sum()}, Duplicates: {df.duplicated().sum()}"
    )

In [ ]:
data_quality_report(orders, "Orders")
data_quality_report(users, "Users")
data_quality_report(events, "Events")
data_quality_report(sessions, "Sessions")
data_quality_report(products, "Products")

In [ ]:
def clean_data(df, name):

    logging.info(f"Cleaning started for {name}")

    before = df.shape

    # remove duplicates
    df = df.drop_duplicates()

    # handle missing values
    for col in df.columns:

        if df[col].dtype == "object":
            df[col] = df[col].fillna("unknown")
        else:
            df[col] = df[col].fillna(df[col].median())

    after = df.shape

    logging.info(f"{name} cleaned | Before: {before} | After: {after}")

    return df

In [ ]:
orders = clean_data(orders, "orders")
users = clean_data(users, "users")
events = clean_data(events, "events")
sessions = clean_data(sessions, "sessions")
products = clean_data(products, "products")

In [ ]:
logging.info("Univariate Analysis - Revenue")

sns.histplot(orders["net_revenue"], bins=30, kde=True)
plt.title("Revenue Distribution")
plt.show()

In [ ]:
logging.info("Event distribution analysis")

events["event_type"].value_counts().plot(kind="bar")
plt.title("Event Distribution")
plt.show()

In [ ]:
logging.info("Bivariate analysis: category vs revenue")

merged = orders.merge(products, on="product_id")

category_sales = merged.groupby("category")["net_revenue"].sum()

category_sales.plot(kind="bar")
plt.title("Revenue by Category")
plt.show()

In [ ]:
logging.info("City revenue analysis")

merged_users = orders.merge(users, on="user_id")

city_sales = merged_users.groupby("city")["net_revenue"].sum()

city_sales.sort_values().plot(kind="barh")
plt.title("Revenue by City")
plt.show()

In [ ]:
logging.info("Event distribution analysis")

events["event_type"].value_counts().plot(kind="bar")
plt.title("Event Distribution")
plt.show()

In [ ]:
logging.info("Funnel analysis started")

funnel = events["event_type"].value_counts()

print(funnel)

In [ ]:
conversion = funnel / funnel["homepage_visit"] * 100

print(conversion)

In [ ]:
logging.info("Time series analysis")

orders["order_date"] = pd.to_datetime(orders["order_date"])

daily_sales = orders.groupby(
    orders["order_date"].dt.date
)["net_revenue"].sum()

daily_sales.plot(figsize=(10,5))
plt.title("Daily Revenue Trend")
plt.show()

In [ ]:
logging.info("Cohort analysis")

cohort_df = orders.merge(users, on="user_id")

cohort_df["order_date"] = pd.to_datetime(cohort_df["order_date"])
cohort_df["sign_up_date"] = pd.to_datetime(cohort_df["sign_up_date"])

cohort_df["cohort_month"] = cohort_df["sign_up_date"].dt.to_period("M")
cohort_df["order_month"] = cohort_df["order_date"].dt.to_period("M")

cohort_table = cohort_df.groupby(
    ["cohort_month", "order_month"]
)["user_id"].nunique().unstack()

print(cohort_table)

In [ ]:
logging.info("Outlier detection started")

Q1 = orders["net_revenue"].quantile(0.25)
Q3 = orders["net_revenue"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = orders[
    (orders["net_revenue"] < lower) |
    (orders["net_revenue"] > upper)
]

logging.info(f"Outliers detected: {len(outliers)}")

print("Outliers:", len(outliers))

In [ ]:
logging.info("Generating business insights")

print("\nTOTAL REVENUE:", orders["net_revenue"].sum())

print("TOP CATEGORY:",
      merged.groupby("category")["net_revenue"].sum().idxmax())

print("TOP CITY:",
      merged_users.groupby("city")["net_revenue"].sum().idxmax())

print("TOTAL ORDERS:", len(orders))

In [ ]:
logging.info("EDA Pipeline Completed Successfully")
logging.info(f"Batch processed: {metadata['batch_id']}")